In [1]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import joblib as jl
from tqdm import tqdm

from functions_synthetic_data_generation import *
from functions_synthetic_data_analysis import *
from functions_hmm import *


plt.style.use('../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline
qt5agg


# Simulations generations

## Parameters

In [9]:
p_cw_reward = 0.8
p_ccw_reward = 0
p_cw_init = 0.5
steps_number = 40
noise_amplitude = 0.1
drift_matrix = np.array([[ 0.05 , -0.05],
                         [ 0    , 0   ]])
drift_init = 0.0

args_dict = {'p_cw_reward': p_cw_reward, 'p_ccw_reward': p_ccw_reward, 'p_cw_init': p_cw_init, 'steps_number': steps_number, 'noise_amplitude': noise_amplitude, 'drift_matrix': drift_matrix, 'drift_init': drift_init}

number_of_simulations_perset = 50

n_simulations_list = [number_of_simulations_perset]*17

start_index = 0

# fit_type = 'aic'
fit_type = 'score'

simulations_folder_path = f'/home/david/Documents/code/DDM_vXbis_synthetic_data_identical_drift'
simulations_folder_path = f'/home/tom/Code/ddm_simulations/simulations_batches_bis/'


## Generation

In [10]:

generate_simulations = False # PARAMS

for i, n_simulations in enumerate(n_simulations_list):
    
    p_cw_init_range = np.linspace(0,1,100)
    
    if not(generate_simulations):

        break

    
    simulations_batch = run_simulations_batch_random_init(args_dict, n_simulations, p_cw_init_range)

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + i+1}.pkl', 'wb') as file:
        pickle.dump(simulations_batch, file)

# HMM fit

## Parameters

In [6]:
n_to_test = np.arange(2,15)
expected_time = np.ceil(len(n_simulations_list)/jl.cpu_count())
print(f'Expected time : {int(expected_time)} t_job')

Expected time : 13 t_job


## Model fit

In [11]:
fit_models = False # PARAMS

if fit_models:

    for i in range(len(n_simulations_list)):

        n_simulations = n_simulations_list[i]

        with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/simulations_batch_{number_of_simulations_perset}_test_{start_index + i+1}.pkl', 'rb') as file:
            synthetic_data = pickle.load(file)

        reformated_training_sequences, reformated_validation_sequences, reformated_training_sequences_lengths, reformated_validation_sequences_lengths = reformat_synthetic_data(synthetic_data)

        fit_output = infer_best_model_score_parallel(reformated_training_sequences, reformated_validation_sequences, 
                                                    reformated_training_sequences_lengths, reformated_validation_sequences_lengths, 
                                                    n_to_test, fix_probability=True, n_features = None, n_fits=200, n_jobs=5)

        save_path = f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + i+1}.pkl'

        with open(save_path, 'wb') as file:
            pickle.dump(fit_output, file)
    
    

In [12]:
example_hmm_number = 1

save_path = f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{example_hmm_number}.pkl'

with open(save_path, 'rb') as file:
    fit_output = pickle.load(file)


## Plots

In [39]:
example_set = 8

with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{example_set}.pkl', 'rb') as file:
    synthetic_data = pickle.load(file)

with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)

model = fit_output['models'][9]
test_data = [synth_data['choices'] for synth_data in synthetic_data]
test_proba = [synth_data['p_cw'] for synth_data in synthetic_data]

########################
### Reformating Data ###
########################


emissionmat = model.emissionprob_
# transmat = model.transmat_

initial_states_arr = np.array(count_initial_state_occurences(model,test_data))

infered_initial_prob_arr = np.array([])

for s in initial_states_arr:

    proba = emissionmat[s,1]
    infered_initial_prob_arr = np.append(infered_initial_prob_arr, proba)

initial_prob_arr = np.array([])

for p_sequence in test_proba:

    # initial_prob_list.append(p_sequence[0])
    initial_prob_arr = np.append(initial_prob_arr,p_sequence[0])

fig = plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)
ax = plt.subplot(row[0,0])

bins = np.linspace( emissionmat[0,1] - (emissionmat[1,1] - emissionmat[0,1])/2,
                    emissionmat[-1,1] + (emissionmat[1,1] - emissionmat[0,1])/2,
                    len(emissionmat)+1)

ax.hist(initial_prob_arr, bins=bins, alpha=0.5, label='True initial proba')
ax.hist(infered_initial_prob_arr, bins=bins, alpha=0.5, label='HMM infered initial proba')

ax.set_yticks(np.arange(0,20,2))
ax.set_xlabel('Starting probability')
ax.set_ylabel('Counts')
ax.legend()

In [41]:
def compute_hmm_slope(transmat, emissionmat):

    slope_list = []
    proba_vect = emissionmat[:,1]

    for i in range(len(transmat)):

        slope = np.sum(transmat[i,:] * proba_vect) - proba_vect[i]
        slope_list.append(slope)

    return slope_list

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

slopes_list = compute_hmm_slope(model.transmat_, model.emissionprob_)

ax1.plot(model.emissionprob_[:,1],slopes_list)
ax1.set_xlabel('State')
ax1.set_ylabel('Average P_n+1(CW)')

print(np.mean(slopes_list))

-0.06906100845473563


In [43]:
mean_slopes_list = []
for i in range(16):

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{i+1}.pkl', 'rb') as file:
        temp_fit_output = pickle.load(file)

    model = temp_fit_output['models'][9]
    test_data = [synth_data['choices'] for synth_data in synthetic_data]
    test_proba = [synth_data['p_cw'] for synth_data in synthetic_data]

    slopes_list = compute_hmm_slope(model.transmat_, model.emissionprob_)

    mean_slopes_list.append(np.mean(slopes_list))

plt.hist(mean_slopes_list)

(array([1., 1., 1., 2., 2., 5., 1., 1., 1., 1.]),
 array([-0.06906101, -0.04672769, -0.02439437, -0.00206104,  0.02027228,
         0.0426056 ,  0.06493892,  0.08727224,  0.10960556,  0.13193889,
         0.15427221]),
 <BarContainer object of 10 artists>)

In [ ]:
# fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
# gs = fig.add_gridspec(1, 1)
# row = gs[:].subgridspec(2, 1, hspace=0.5)

# ax1 = plt.subplot(row[0,0])

# best_states_nbr_list = []

# for i in range(len(n_simulations_list)):

#     save_path = f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{example_hmm_number}.pkl'

#     with open(save_path, 'rb') as file:
#             fit_output = pickle.load(file)

#     ax1.plot(n_to_test, fit_output['scores'], alpha=0.2)
#     # ax1.scatter(len(fit_output[i][0].transmat_), fit_output[i][1], marker='+', alpha=0.2)

#     best_states_nbr_list.append(n_to_test[np.argmax(fit_output['scores'])])

# ax1.set_xticks(n_to_test)
# ax1.set_xlabel("Number of states")
# ax1.set_ylabel("log fit score")

# ax2 = plt.subplot(row[1,0])

# ax2.hist(best_states_nbr_list, bins=n_to_test, align='left')
# ax2.set_xlim([1,n_to_test[-1]+1])
# ax2.set_xlabel('Number of states')
# ax2.set_ylabel('Number of HMM')


Text(0, 0.5, 'Number of HMM')

In [15]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

best_states_nbr_list = []

for i in range(len(n_simulations_list)):

    save_path = f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + i+1}.pkl'

    with open(save_path, 'rb') as file:
            fit_output = pickle.load(file)

    ax1.plot(n_to_test, fit_output['scores'], alpha=0.2)
    # ax1.scatter(len(fit_output[i][0].transmat_), fit_output[i][1], marker='+', alpha=0.2)

    best_states_nbr_list.append(n_to_test[np.argmax(fit_output['scores'])])

ax1.set_xticks(n_to_test)
ax1.set_xlabel("Number of states")
ax1.set_ylabel("log fit score")

ax2 = plt.subplot(row[1,0])

ax2.hist(best_states_nbr_list, bins=n_to_test, align='left')
ax2.set_xlim([1,n_to_test[-1]+1])
ax2.set_xlabel('Number of states')
ax2.set_ylabel('Number of HMM')


Text(0, 0.5, 'Number of HMM')

In [ ]:
# def compute_hmm_increase_proba(transmat,emissionmat):

#     up_proba_list = []

#     for i, row in enumerate(transmat):

#         up_proba = np.sum(row[i+1:])
#         up_proba_list.append(up_proba)

#     return emissionmat[:,1], np.array(up_proba_list)

# with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/simulations_batch_{number_of_simulations_perset}_test_{1}.pkl', 'rb') as file:
#     synthetic_data = pickle.load(file)

# with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{1}.pkl', 'rb') as file:
#     fit_output = pickle.load(file)

# model = fit_output['models'][10]

# # reformated_matrices = reformat_hmm(synthetic_data,model)

# transmat = model.transmat_
# emissionmat = model.emissionprob_

# res = compute_hmm_increase_proba(transmat,emissionmat)

# fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
# gs = fig.add_gridspec(1, 1)
# row = gs[:].subgridspec(1, 1, hspace=0.5)

# ax1 = plt.subplot(row[0,0])

# ax1.scatter(res[0],res[1])
# ax1.set_xlabel('P(CW)')
# ax1.set_ylabel('P_up')


Text(0, 0.5, 'P_up')

In [25]:
index_nb_of_states = 10

fixed_state_models_list = []

for j in range(len(n_simulations_list)):

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index+j+1}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    transmat_to_add = fit_output['models'][index_nb_of_states].transmat_
    fixed_state_models_list.append(transmat_to_add)

average_transmat = np.mean(fixed_state_models_list, axis=0)
var_transmat = np.var(fixed_state_models_list, axis=0)

new_mat = average_transmat

for _ in range(100):

    new_mat = np.matmul(new_mat,average_transmat)

# pop_vect = np.vecmat(np)


fig=plt.figure(figsize=(3.5, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[0].subgridspec(1,2)
ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[0,1])

# ax1.imshow(average_transmat, vmin=0, vmax=1)
ax1.imshow(var_transmat, vmin=0, vmax=1)
ax1.set_xticks(np.arange(len(average_transmat)))
ax1.set_yticks(np.arange(len(average_transmat)))

ax2.imshow(average_transmat, vmin=0, vmax=1)
ax2.set_xticks(np.arange(len(average_transmat)))
ax2.set_yticks(np.arange(len(average_transmat)))


# Drift Estimation

## Generating "theoretical" average probability sequences

## Minimizing mean square error

In [ ]:
generate_theoretical_sequences = False # PARAM

if generate_theoretical_sequences:

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index +1}.pkl', 'rb') as file:
                synthetic_data = pickle.load(file)

    args = [synthetic_data[0]['parameters']] + [5000] + [p_cw_init_range]
    drift_range = np.linspace(0.01,0.2,200)

    # test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts(drift_range, args)
    test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts_random_init(drift_range, args)

    with open(f'{simulations_folder_path}/test_average_probability_sequences.pkl', 'wb') as file:
        pickle.dump([drift_range, test_average_probability_sequences], file)

else:
 
    with open(f'{simulations_folder_path}/test_average_probability_sequences.pkl', 'rb') as file:
        drift_range, test_average_probability_sequences = pickle.load(file)


100%|██████████| 200/200 [16:50<00:00,  5.05s/it]


In [19]:
save_result = True # PARAM
fit_type = 'score' # 'score'


# for index, _ in enumerate(tqdm(n_simulations_list)):
for index, _ in enumerate(n_simulations_list):

    average_proba_sequence_hmm = []

    ##############################
    ### Loading Data and Model ###
    ##############################

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        synthetic_data = pickle.load(file)

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + index+1}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][np.argmax(fit_output['scores'])]

    ########################
    ### Reformating Data ###
    ########################

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    emissionmat = model.emissionprob_
    transmat = model.transmat_
    initial_state_distri = compute_initial_state_list_distri(model,test_data)

    ####################
    ### Computations ###
    ####################

    mat_i = copy.deepcopy(transmat)

    for i in range(len(test_data[0])):

        mat_i = np.matmul(mat_i,transmat)
        res = np.matmul(initial_state_distri,mat_i)*emissionmat[:,1]
    
        average_proba_sequence_hmm.append(np.sum(res))

    mse_list = []

    # for test_msd_sequence in tqdm(test_msd_sequence_list, leave=False):
    for test_average_probability_sequence in test_average_probability_sequences:
    
        mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(mse_list)
    recovered_drift = drift_range[np.where(mse_list==min_mse)[0]]

    if not(save_result):

        continue

    ##############################
    ### Saving Recovered drift ###
    ##############################
    
    with open(f'{simulations_folder_path}/n_{n_simulations}/recovered_drift_{n_simulations}_{start_index + index+1}.pkl', 'wb') as file:
        pickle.dump([test_average_probability_sequences,recovered_drift], file)


In [20]:
recovered_drift_list = []

for index, _ in enumerate(n_simulations_list):

    with open(f'{simulations_folder_path}/n_{n_simulations}/recovered_drift_{n_simulations}_{start_index + index+1}.pkl', 'rb') as file:
        _, recovered_drift = pickle.load(file)
    
    recovered_drift_list.append(recovered_drift[0])

# Plots

In [21]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(recovered_drift_list, bins=np.linspace(0.01,0.2,51))
bin_width = histo[1][1] - histo[1][0]
ax1.stairs(histo[0], histo[1], alpha=0.5, fill=True, label = 'Recovered drift probability density')

ax1.axvline(0.05, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{len(n_simulations_list)} sets of {number_of_simulations_perset} simulations of length {steps_number}')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Probability density')

ax1.legend()